# Minimal working example using BERT (`distBERT` model)

In [2]:
#!pip install pandas torch transformers faiss-cpu

## Setup

In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
import faiss

## Pseudo data creation

In [10]:
# ----------------------------
# 1) Data 
# ----------------------------

items = pd.read_csv('/Users/Maya/Downloads/items.csv')
events = pd.read_csv('/Users/Maya/Downloads/events.csv')

events.head()


,user_id,item_id,ts,event_type,session_id,device,dwell_seconds,price_at_event,category_at_event
0,u1,i2,2026-01-01 07:47:00,click,s_u1_20260101_1,tablet,56,129.00,electronics
1,u1,i13,2026-01-04 08:01:00,view,s_u1_20260104_2,tablet,7,149.99,electronics
2,u1,i2,2026-01-07 14:28:00,click,s_u1_20260107_3,mobile,72,129.00,electronics
3,u1,i13,2026-01-07 19:06:00,view,s_u1_20260107_1,tablet,67,149.99,electronics
4,u1,i1,2026-01-10 08:42:00,add_to_cart,s_u1_20260110_1,desktop,46,199.99,electronics


In [12]:
items.head()

,item_id,title,description,category,brand,price,tags
0,i1,Noise Cancelling Headphones,Wireless noise-cancelling headphones with 30-h...,electronics,SoundPeak,199.99,"audio,wireless,travel,premium"
1,i2,Mechanical Keyboard,"Mechanical keyboard with RGB backlighting, hot...",electronics,KeyForge,129.00,"keyboard,gaming,productivity,desk"
2,i3,Running Shoes,Running shoes designed for long distance comfo...,sports,StrideLab,110.00,"running,fitness,outdoors,comfort"
3,i4,Vegetarian Cookbook,"Cookbook featuring quick vegetarian recipes, p...",books,HomeTable Press,24.99,"cooking,vegetarian,recipes,home"
4,i5,Fitness Smartwatch,"Smartwatch with heart rate monitoring, GPS, sl...",electronics,PulsePath,249.00,"wearables,fitness,gps,health"


In [14]:

# make sure timestamp is datetime
events["ts"] = pd.to_datetime(events["ts"])

## BERT encoder (What makes BERT work?)

In [17]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "distilbert-base-uncased"  # for illustration, 66M model

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Evaluate the BERT encoder**

In [20]:
# ----------------------------
# 2) BERT encode helper
# ----------------------------

# we want to go from text to tokens 
# when we embed we can decide our max length and then decide how long we want the vector to be 
# as you increase embedding size you are capturing more and more information 


# turn off gradient calculation (no back propagation in using BERT)
@torch.no_grad()
def bert_embed(texts, max_len=128):
    batch = tokenizer(
        texts, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
    )
    batch = {k: v.to(DEVICE) for k, v in batch.items()}
    out = encoder(**batch)
    cls = out.last_hidden_state[:, 0]          # first column [CLS]-like token for classification
    emb = F.normalize(cls, dim=-1)             # normalization
    return emb.cpu().numpy().astype("float32") # size (B, 768)


## Embedding items into vectors (for comparisons)

In [23]:
# ----------------------------
# 3) Offline job: item embeddings
# ----------------------------

items["text"] = (
    items["title"].fillna("") + ". " +
    items["description"].fillna("") + ". " +
    items["category"].fillna("") + ". " +
    items["brand"].fillna("") + ". " +
    items["tags"].fillna("")
)
item_vecs = bert_embed(items["text"].tolist())
item_id_list = items["item_id"].tolist()

# Build ANN index (inner product works with normalized vectors)
index = faiss.IndexFlatIP(item_vecs.shape[1])
index.add(item_vecs)


## What user-specific data are there?

In [40]:
# ----------------------------
# 4) Feature builder: user text from last N clicks
# ----------------------------

## include more info about whether the customer added to card, clicked, or viewed and how long the customer dwells on the item. 
## adding to cart is better than clicking is better than viewing. the more time they spend the better 

def build_user_text(user_id, events, items, N=3):
    hist = (events[events["user_id"] == user_id]
            .sort_values("ts")
            .tail(N))
    if hist.empty:
        return "no history", set()
    text_lookup = items.set_index("item_id")["text"]
    text = []
    seen = set()

    for _, row in hist.iterrows():
        iid = row["item_id"]
        item_text = items.set_index("item_id").loc[iid, "text"]

        repeats = 1
        if row["event_type"] == "click":
            repeats = 2
        elif row["event_type"] == "add_to_cart":
            repeats = 3

        if row["dwell_seconds"] >= 60:
            repeats += 1

        text.extend([item_text] * repeats)
        seen.add(iid)

    return " ".join(text), seen

## How to recommend "similar" item with BERT?

In [43]:
# ----------------------------
# 5) "What to recommend" function
# ----------------------------
def recommend(user_id, k=3):
    user_text, seen = build_user_text(user_id, events, items, N=3)
    u = bert_embed([user_text])  # (1, 768)
    scores, idx = index.search(u, k + len(seen))  # keep track of what was seen by the user
    recs = []
    for j in idx[0]:
        iid = item_id_list[j]
        if iid not in seen:
            recs.append(iid)
        if len(recs) == k:
            break
    return recs

In [45]:

for u in ["u1","u2","u3"]:
    print(u, "->", recommend(u, k=3))


u1 -> ['i6', 'i13', 'i14']
u2 -> ['i18', 'i6', 'i13']
u3 -> ['i8', 'i4', 'i18']


### Sanity Check

In [48]:
text, seen = build_user_text("u1", events, items)
print("USER TEXT:")
print(text)
print("SEEN:", seen)

recs = recommend("u1", k=5)
print("RECOMMENDED IDS:", recs)

print(items[items["item_id"].isin(list(seen) + recs)][["item_id", "title", "category", "brand"]])

USER TEXT:
Fitness Smartwatch. Smartwatch with heart rate monitoring, GPS, sleep tracking, and guided workout summaries. electronics. PulsePath. wearables,fitness,gps,health Fitness Smartwatch. Smartwatch with heart rate monitoring, GPS, sleep tracking, and guided workout summaries. electronics. PulsePath. wearables,fitness,gps,health Cycling Gloves. Breathable cycling gloves with gel padding for weekend rides and indoor spin classes. sports. TrailSip. cycling,fitness,outdoors,comfort Cycling Gloves. Breathable cycling gloves with gel padding for weekend rides and indoor spin classes. sports. TrailSip. cycling,fitness,outdoors,comfort Cycling Gloves. Breathable cycling gloves with gel padding for weekend rides and indoor spin classes. sports. TrailSip. cycling,fitness,outdoors,comfort Running Shoes. Running shoes designed for long distance comfort, arch support, and all-weather road grip. sports. StrideLab. running,fitness,outdoors,comfort Running Shoes. Running shoes designed for long